In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/processed/heart_imputed.csv")
print(df.shape)
print("Blanks:", df.isna().sum().sum())

# Leakage safeguard: set the label and hospital aside
labels = df[["HeartDisease", "source"]].copy()
features = df.drop(columns=["HeartDisease", "source"])

assert "HeartDisease" not in features.columns
assert "source" not in features.columns
print(features.shape)

(918, 13)
Blanks: 0
(918, 11)


In [3]:
from sklearn.preprocessing import StandardScaler

cont_cols = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak"]

X = pd.DataFrame(index=features.index)

# 1. Standardise the continuous columns
scaler = StandardScaler()
X[cont_cols] = scaler.fit_transform(features[cont_cols])

# 2. Yes/no variables: one column each
X["FastingBS"] = features["FastingBS"].astype(int)
X["Sex_M"] = (features["Sex"] == "M").astype(int)
X["ExerciseAngina_Y"] = (features["ExerciseAngina"] == "Y").astype(int)

# 3. Variables with 3+ categories: one column per category
multi = pd.get_dummies(features[["ChestPainType", "RestingECG", "ST_Slope"]], dtype=int)
X = pd.concat([X, multi], axis=1)

print(X.shape)
print(X.columns.tolist())

# Save it: every pipeline (PCA, autoencoder, Gower checks) will start from this same table
X.to_csv("data/processed/X_model.csv", index=False)

(918, 18)
['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak', 'FastingBS', 'Sex_M', 'ExerciseAngina_Y', 'ChestPainType_ASY', 'ChestPainType_ATA', 'ChestPainType_NAP', 'ChestPainType_TA', 'RestingECG_LVH', 'RestingECG_Normal', 'RestingECG_ST', 'ST_Slope_Down', 'ST_Slope_Flat', 'ST_Slope_Up']


In [4]:
from sklearn.decomposition import PCA

# Fit PCA with all 18 components first, to see how the variation is shared out
pca_full = PCA(random_state=42)
pca_full.fit(X)

explained = pca_full.explained_variance_ratio_   # share of variation per PC
cumulative = np.cumsum(explained)                # running total

for i, (e, c) in enumerate(zip(explained, cumulative), start=1):
    print(f"PC{i}: {e:.3f}   cumulative: {c:.3f}")

PC1: 0.270   cumulative: 0.270
PC2: 0.139   cumulative: 0.409
PC3: 0.128   cumulative: 0.537
PC4: 0.117   cumulative: 0.654
PC5: 0.090   cumulative: 0.744
PC6: 0.046   cumulative: 0.790
PC7: 0.041   cumulative: 0.832
PC8: 0.039   cumulative: 0.871
PC9: 0.027   cumulative: 0.898
PC10: 0.025   cumulative: 0.924
PC11: 0.021   cumulative: 0.944
PC12: 0.020   cumulative: 0.964
PC13: 0.016   cumulative: 0.980
PC14: 0.012   cumulative: 0.992
PC15: 0.008   cumulative: 1.000
PC16: 0.000   cumulative: 1.000
PC17: 0.000   cumulative: 1.000
PC18: 0.000   cumulative: 1.000


In [5]:
# Fit PCA again, keeping 7 components (first to reach >= 80% variance)
pca = PCA(n_components=7, random_state=42)
Z_pca = pca.fit_transform(X)

print(Z_pca.shape)
print("Variance explained:", round(pca.explained_variance_ratio_.sum() * 100, 1), "%")

# Save the PCA representation for clustering later
Z_pca_df = pd.DataFrame(Z_pca, columns=[f"PC{i}" for i in range(1, 8)])
Z_pca_df.to_csv("data/processed/Z_pca.csv", index=False)
print("Saved.")

(918, 7)
Variance explained: 83.2 %
Saved.


In [6]:
# 1. Each patient's scores (the new 7 columns)
print(Z_pca_df.head())

# 2. The PC1 recipe: how much each original column contributes
recipe = pd.Series(pca.components_[0], index=X.columns)
print(recipe.sort_values(key=abs, ascending=False).round(2))

        PC1       PC2       PC3       PC4       PC5       PC6       PC7
0 -2.034775  1.517059  0.299777 -0.154429 -0.648111  0.385701 -0.253332
1 -0.360269 -0.168573  1.988580 -0.342378 -0.740566  0.384667  1.169785
2 -0.468868 -0.050133 -1.374414 -0.760514 -1.701761 -0.838413  0.068450
3  0.766216 -0.924060 -0.098345  0.253870 -1.292676  0.251656  0.053794
4  0.331890  0.725283 -0.266116 -1.288112 -0.592567  0.464200  0.922558
Age                  0.52
MaxHR               -0.49
Oldpeak              0.43
RestingBP            0.34
ST_Slope_Up         -0.21
ExerciseAngina_Y     0.17
ST_Slope_Flat        0.17
ChestPainType_ASY    0.16
Cholesterol          0.15
ChestPainType_ATA   -0.11
RestingECG_Normal   -0.08
FastingBS            0.06
RestingECG_ST        0.06
Sex_M                0.05
ChestPainType_NAP   -0.05
ST_Slope_Down        0.04
RestingECG_LVH       0.02
ChestPainType_TA    -0.00
dtype: float64


In [7]:
features.iloc[0]

Age                 40.0
Sex                    M
ChestPainType        ATA
RestingBP          140.0
Cholesterol        289.0
FastingBS              0
RestingECG        Normal
MaxHR              172.0
ExerciseAngina         N
Oldpeak              0.0
ST_Slope              Up
Name: 0, dtype: object

## Modelling table and PCA

**Input:** `data/processed/heart_imputed.csv`. HeartDisease and source set aside (assert checks).

### Modelling table (`X_model.csv`, 918 × 18)
- Continuous (standardised, mean 0, SD 1): Age, RestingBP, Cholesterol, MaxHR, Oldpeak
- Binary (one 0/1 column each, to avoid double-counting): FastingBS, Sex_M, ExerciseAngina_Y
- Multi-category (one column per category): ChestPainType (4), RestingECG (3), ST_Slope (3)
- Same table will be the input for the autoencoder, so the comparison is fair.

### PCA
- Rule fixed in advance (no label): keep the fewest components reaching ≥ 80% variance (target window 80–85%).
- PC1 27.0%; PC6 cumulative 79.0%; **PC7 cumulative 83.2% → 7 components kept**; PC8 87.1%.
- Compression: 18 → 7 columns, keeping 83.2% of the variation.
- PC16–18 explain 0%: one redundant column in each multi-category group (ChestPainType, RestingECG, ST_Slope).
- PC1 recipe: Age +0.52, MaxHR −0.49 → high PC1 = older with lower max heart rate (consistent with the known age–MaxHR relationship).
- Check: patient 0 (age 40, MaxHR 172) has PC1 = −2.03, as expected.
- Note: continuous columns dominate the leading PCs, partly because they were standardised (SD 1) while binary columns have smaller spread. One motivation for the Gower reference track.

**Output:** `data/processed/Z_pca.csv` (918 × 7).